In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoProcessor, LlavaForConditionalGeneration
from PIL import Image
from typing import Optional

model_id = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(model_id)
print("Processor loaded successfully!")

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)
print("Model loaded successfully!")

/venv/vlm-llava/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Processor loaded successfully!


Loading checkpoint shards: 100%|██████████| 3/3 [00:02<00:00,  1.15it/s]

Model loaded successfully!


In [2]:
def add_diffusion_noise(pixel_values: torch.Tensor, noise_step: int = 500):
    """
    VCD-style image corruption.
    pixel_values: [B, C, H, W], already preprocessed by LLaVA processor.
    """
    num_steps = 1000
    noise_step = max(0, min(int(noise_step), num_steps - 1))

    device = pixel_values.device

    betas = torch.linspace(-6, 6, num_steps, device=device, dtype=torch.float32)
    betas = torch.sigmoid(betas) * (0.5e-2 - 1e-5) + 1e-5

    alphas = 1.0 - betas
    alphas_prod = torch.cumprod(alphas, dim=0)

    sqrt_alpha_prod = torch.sqrt(alphas_prod[noise_step]).to(pixel_values.dtype)
    sqrt_one_minus_alpha_prod = torch.sqrt(1.0 - alphas_prod[noise_step]).to(pixel_values.dtype)

    noise = torch.randn_like(pixel_values)
    noisy_pixel_values = sqrt_alpha_prod * pixel_values + sqrt_one_minus_alpha_prod * noise

    return noisy_pixel_values

In [3]:
def top_k_top_p_filtering(
    logits: torch.Tensor,
    top_k: Optional[int] = None,
    top_p: Optional[float] = None,
    filter_value: float = -float("inf"),
):
    """
    Simple HuggingFace-style top-k / nucleus filtering.
    logits: [B, vocab]
    """
    if top_k is not None and top_k > 0:
        top_k = min(top_k, logits.size(-1))
        kth_values = torch.topk(logits, top_k, dim=-1).values[:, -1].unsqueeze(-1)
        logits = logits.masked_fill(logits < kth_values, filter_value)

    if top_p is not None and 0.0 < top_p < 1.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
        sorted_probs = F.softmax(sorted_logits, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

        sorted_indices_to_remove = cumulative_probs > top_p

        # Keep at least the first token above threshold.
        sorted_indices_to_remove[:, 1:] = sorted_indices_to_remove[:, :-1].clone()
        sorted_indices_to_remove[:, 0] = False

        indices_to_remove = torch.zeros_like(logits, dtype=torch.bool)
        indices_to_remove.scatter_(dim=-1, index=sorted_indices, src=sorted_indices_to_remove)
        logits = logits.masked_fill(indices_to_remove, filter_value)

    return logits

In [4]:
from typing import Optional
import torch
import torch.nn.functional as F


@torch.inference_mode()
def vcd_generate_batch(
    model,
    processor,
    images,
    question: str = "Describe this image.",
    max_new_tokens: int = 256,
    cd_alpha: float = 1.0,
    cd_beta: float = 0.1,
    noise_step: int = 500,
    temperature: float = 1.0,
    top_p: Optional[float] = 0.9,
    top_k: Optional[int] = None,
    do_sample: bool = False,
):
    model.eval()

    single_input = not isinstance(images, list)

    if single_input:
        images = [images]

    batch_size = len(images)

    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype

    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"

    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    prompt = f"USER: <image>\n{question}\nASSISTANT:"
    prompts = [prompt for _ in range(batch_size)]

    inputs = processor(
        text=prompts,
        images=images,
        return_tensors="pt",
        padding=True,
    )

    processor.tokenizer.padding_side = old_padding_side

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    pixel_values = inputs["pixel_values"].to(device=device, dtype=dtype)
    pixel_values_cd = add_diffusion_noise(pixel_values, noise_step=noise_step)

    generated_ids = input_ids.clone()
    prompt_len = input_ids.shape[1]

    past_key_values = None
    past_key_values_cd = None

    next_input_ids = input_ids
    next_input_ids_cd = input_ids

    eos_token_id = model.generation_config.eos_token_id

    if eos_token_id is None:
        eos_token_ids = []
    elif isinstance(eos_token_id, int):
        eos_token_ids = [eos_token_id]
    else:
        eos_token_ids = list(eos_token_id)

    fallback_token_id = (
        eos_token_ids[0]
        if len(eos_token_ids) > 0
        else processor.tokenizer.eos_token_id
    )

    if fallback_token_id is None:
        fallback_token_id = processor.tokenizer.pad_token_id

    if fallback_token_id is None:
        fallback_token_id = 0

    finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

    for step in range(max_new_tokens):
        if step == 0:
            outputs = model(
                input_ids=next_input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                use_cache=True,
                return_dict=True,
            )

            outputs_cd = model(
                input_ids=next_input_ids_cd,
                attention_mask=attention_mask,
                pixel_values=pixel_values_cd,
                use_cache=True,
                return_dict=True,
            )

        else:
            outputs = model(
                input_ids=next_input_ids,
                attention_mask=attention_mask,
                past_key_values=past_key_values,
                use_cache=True,
                return_dict=True,
            )

            outputs_cd = model(
                input_ids=next_input_ids_cd,
                attention_mask=attention_mask,
                past_key_values=past_key_values_cd,
                use_cache=True,
                return_dict=True,
            )

        logits = outputs.logits[:, -1, :]
        logits_cd = outputs_cd.logits[:, -1, :]

        past_key_values = outputs.past_key_values
        past_key_values_cd = outputs_cd.past_key_values

        # VCD logits
        vcd_logits = (1.0 + cd_alpha) * logits - cd_alpha * logits_cd

        # Adaptive plausibility constraint
        cutoff = torch.log(
            torch.tensor(cd_beta, device=logits.device, dtype=logits.dtype)
        ) + logits.max(dim=-1, keepdim=True).values

        vcd_logits = vcd_logits.masked_fill(logits < cutoff, -float("inf"))

        if temperature is not None and temperature > 0 and temperature != 1.0:
            vcd_logits = vcd_logits / temperature

        vcd_logits = top_k_top_p_filtering(
            vcd_logits,
            top_k=top_k,
            top_p=top_p,
        )

        # Finished samples keep producing fallback token
        vcd_logits[finished, :] = -float("inf")
        vcd_logits[finished, fallback_token_id] = 0.0

        if do_sample:
            probs = F.softmax(vcd_logits, dim=-1)

            probs = torch.nan_to_num(
                probs,
                nan=0.0,
                posinf=0.0,
                neginf=0.0,
            )

            row_sums = probs.sum(dim=-1, keepdim=True)

            bad_rows = row_sums.squeeze(-1) == 0
            if bad_rows.any():
                probs[bad_rows, fallback_token_id] = 1.0
                row_sums = probs.sum(dim=-1, keepdim=True)

            probs = probs / row_sums
            next_token = torch.multinomial(probs, num_samples=1)

        else:
            next_token = torch.argmax(vcd_logits, dim=-1, keepdim=True)

        next_token = torch.where(
            finished.unsqueeze(-1),
            torch.full_like(next_token, fallback_token_id),
            next_token,
        )

        generated_ids = torch.cat([generated_ids, next_token], dim=-1)

        attention_mask = torch.cat(
            [
                attention_mask,
                torch.ones(
                    (batch_size, 1),
                    dtype=attention_mask.dtype,
                    device=attention_mask.device,
                ),
            ],
            dim=-1,
        )

        next_input_ids = next_token
        next_input_ids_cd = next_token

        if len(eos_token_ids) > 0:
            newly_finished = torch.zeros_like(finished)

            for eid in eos_token_ids:
                newly_finished |= next_token.squeeze(-1).eq(eid)

            finished |= newly_finished

            if finished.all():
                break

    new_tokens = generated_ids[:, prompt_len:]

    answers = processor.batch_decode(
        new_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    answers = [ans.strip() for ans in answers]

    if single_input:
        return answers[0]

    return answers

In [5]:
import json
import gc
from pathlib import Path

import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from datasets import load_dataset


processor.tokenizer.padding_side = "left"

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token


def batch_list(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


def clean_output(text):
    text = text.strip()

    if "ASSISTANT:" in text:
        text = text.split("ASSISTANT:", 1)[-1].strip()

    return text.strip()


def save_json(rows, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2, ensure_ascii=False)

In [10]:
CACHE_DIR = None

BATCH_SIZE = 24
MAX_NEW_TOKENS = 256

PROMPT = "Describe this image."

OUTPUT_JSON = "../results/infer/openchair/llava15/vcd.json"
OUTPUT_CSV = "../results/infer/openchair/llava15/vcd.csv"

CD_ALPHA = 1.0
CD_BETA = 0.1
NOISE_STEP = 500
TEMPERATURE = 1.0
TOP_P = 0.9
TOP_K = None
DO_SAMPLE = False

In [11]:
och_dataset = load_dataset(
    "moranyanuka/OpenCHAIR",
    cache_dir=CACHE_DIR
)["test"]

print(och_dataset)
print("Columns:", och_dataset.column_names)
print("Num samples:", len(och_dataset))

Dataset({
    features: ['image', 'text'],
    num_rows: 4863
})
Columns: ['image', 'text']
Num samples: 4863


In [ ]:
model.eval()

indices = list(range(len(och_dataset)))
results = []

# Resume support
if Path(OUTPUT_JSON).exists():
    with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
        results = json.load(f)

    done_indices = set(int(row["index"]) for row in results)
    indices = [idx for idx in indices if idx not in done_indices]

    print(f"Resume mode: loaded {len(results)} existing captions.")
    print(f"Remaining samples: {len(indices)}")

batches = list(batch_list(indices, BATCH_SIZE))

for batch_indices in tqdm(batches, desc="OpenCHAIR VCD inference"):
    images = []
    valid_indices = []

    for idx in batch_indices:
        image = och_dataset[idx]["image"]

        if isinstance(image, Image.Image):
            image = image.convert("RGB")
        else:
            image = Image.open(image).convert("RGB")

        images.append(image)
        valid_indices.append(idx)

    outputs = vcd_generate_batch(
        model=model,
        processor=processor,
        images=images,
        question=PROMPT,
        max_new_tokens=MAX_NEW_TOKENS,
        cd_alpha=CD_ALPHA,
        cd_beta=CD_BETA,
        noise_step=NOISE_STEP,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        do_sample=DO_SAMPLE,
    )

    for idx, output in zip(valid_indices, outputs):
        caption = clean_output(output)

        results.append({
            "index": int(idx),
            "generated_caption": caption,
        })

    # Save after each batch
    results = sorted(results, key=lambda x: x["index"])
    save_json(results, OUTPUT_JSON)

    del images
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"Saved {len(results)} captions to {OUTPUT_JSON}")

OpenCHAIR VCD inference:   0%|          | 0/203 [00:00<?, ?it/s]

In [ ]:
with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
    rows = json.load(f)

rows = sorted(rows, key=lambda x: x["index"])

df = pd.DataFrame({
    "generated_caption": [row["generated_caption"] for row in rows]
})

Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved CSV to {OUTPUT_CSV}")
df.head()